In [4]:
from numpy import array, pi, log, exp, sqrt, conj, concatenate, linspace, linalg
from scipy.integrate import solve_ivp
from scipy.interpolate import CubicSpline

######################################################
######################################################
######################################################

def eos(rho,vec_rho,vec_p):
    for i in range(0,len(vec_rho)-1):
        if(rho<=vec_rho[0]):
            print('ERROR_1')
            break
        elif((rho>vec_rho[i])and(rho<=vec_rho[i+1])):
            dlogpdlogrho=log(vec_p[i+1]/vec_p[i])/log(vec_rho[i+1]/vec_rho[i])
            p=vec_p[i]*((rho/vec_rho[i])**dlogpdlogrho)
            Gamma=((rho+p)/rho)*dlogpdlogrho
        elif(rho>vec_rho[-1]):
            print('ERROR_2')
            break
    return(p,Gamma)

def eos_inv(p,vec_p,vec_rho):
    for i in range(0,len(vec_p)-1):
        if(p<=vec_p[0]):
            print('ERROR_3')
            break
        elif((p>vec_p[i])and(p<=vec_p[i+1])):
            dlogpdlogrho=log(vec_p[i+1]/vec_p[i])/log(vec_rho[i+1]/vec_rho[i])
            rho=vec_rho[i]*((p/vec_p[i])**(1.0/dlogpdlogrho))
            Gamma=((rho+p)/rho)*dlogpdlogrho
        elif(p>vec_p[-1]):
            print('ERROR_4')
            break
    return(rho,Gamma)

def eos_func(eos_str):

    #########################
    # eos arrays: p and rho #
    #########################

    file_eos=open(eos_str+'.txt','r')

    vec_p_tab=[]
    vec_rho_tab=[]
    for line in file_eos:
        vec_p_tab.append(float(line.split()[1]))
        vec_rho_tab.append(float(line.split()[2]))
    vec_p_tab=array(vec_p_tab)
    vec_rho_tab=array(vec_rho_tab)

    vec_p=vec_p_tab*7.4237e-19
    vec_rho=vec_rho_tab*7.4237e-19

    return(vec_p,vec_rho)

def tov_func(eos_func_array,rho_0_cgs):

    def dmdr(r,rho):
        mp=4.0*pi*(r**2.0)*rho
        return(mp)

    def dpdr(r,p,m,rho):
        pp=-((rho+p)*(m+(4.0*pi*(r**3.0)*p)))/(r*(r-(2.0*m)))
        return(pp) 

    def dnudr(r,m,p):
        nup=(2.0*(m+(4.0*pi*(r**3.0)*p)))/(r*(r-(2.0*m)))
        return(nup)

    vec_p,vec_rho=eos_func_array

    ##############################################
    # creates lists for: r, m, nu, p, rho, Gamma #
    ##############################################

    rv=[]
    mv=[]
    nuv=[]
    pv=[]
    rhov=[]
    Gammav=[]

    ########################
    # values at the center #
    ########################

    rho_0=rho_0_cgs*7.4237e-19
    eos_vars=eos(rho_0,vec_rho,vec_p)
    p_0=eos_vars[0]
    Gamma_0=eos_vars[1]
    r_0=0.0
    m_0=0.0
    nu_0=0.0

    rv.append(r_0)
    mv.append(m_0)
    nuv.append(nu_0)
    pv.append(p_0)
    rhov.append(rho_0)
    Gammav.append(Gamma_0)

    #############################
    # second order coefficients #
    #############################

    m_3=4.0*pi*rho_0
    nu_2=((8.0*pi)/3.0)*(rho_0+(3.0*p_0))
    p_2=-(((4.0*pi)/3.0)*(rho_0+p_0)*(rho_0+(3.0*p_0)))
    rho_2=(p_2*(rho_0+p_0))/(Gamma_0*p_0)

    ######################
    # minimum r and step #
    ######################

    r_min=1.0e-03
    dr=1.0e-03

    #######################
    # values at minimum r #
    #######################

    r=r_min
    m=(1.0/3.0)*m_3*(r**3.0)
    nu=nu_0+((1.0/2.0)*nu_2*(r**2.0))
    p=p_0+((1.0/2.0)*p_2*(r**2.0))
    rho=rho_0+((1.0/2.0)*rho_2*(r**2.0))
    Gamma=eos(rho_0,vec_rho,vec_p)[1]

    rv.append(r)
    mv.append(m)
    nuv.append(nu)
    pv.append(p)
    rhov.append(rho)
    Gammav.append(Gamma)

    ############
    # tov loop #
    ############

    while(p>=vec_p[0]):

        s1=dmdr(r,rho)
        s2=dmdr(r+(dr/2.0),rho)
        s3=dmdr(r+(dr/2.0),rho)
        s4=dmdr(r+dr,rho)

        t1=dpdr(r,p,m,rho)
        t2=dpdr(r+(dr/2.0),p+(t1*(dr/2.0)),m,rho)
        t3=dpdr(r+(dr/2.0),p+(t2*(dr/2.0)),m,rho)
        t4=dpdr(r+dr,p+(t3*dr),m,rho)

        u1=dnudr(r,m,p)
        u2=dnudr(r+(dr/2.0),m,p)
        u3=dnudr(r+(dr/2.0),m,p)
        u4=dnudr(r+dr,m,p)

        p=p+((t1+(2.0*t2)+(2.0*t3)+t4)*(dr/6.0))
        if(p<vec_p[0]):
            p=p-((t1+(2.0*t2)+(2.0*t3)+t4)*(dr/6.0))
            break
        m=m+((s1+(2.0*s2)+(2.0*s3)+s4)*(dr/6.0))
        nu=nu+((u1+(2.0*u2)+(2.0*u3)+u4)*(dr/6.0))

        eos_inv_vars=eos_inv(p,vec_p,vec_rho)
        rho=eos_inv_vars[0]
        Gamma=eos_inv_vars[1]
        r=r+dr

        rv.append(r)
        mv.append(m)
        nuv.append(nu)
        pv.append(p)
        rhov.append(rho)
        Gammav.append(Gamma)

    ##############################
    # transforms lists in arrays #
    ##############################

    rv=array(rv)
    mv=array(mv)
    nuv=array(nuv)
    pv=array(pv)
    rhov=array(rhov)
    Gammav=array(Gammav)

    ###############################
    # determines nu at the center #
    ###############################

    r_s=r
    m_s=m
    nu_s=log(1.0-((2.0*m_s)/r_s))
    nu_0=nu_s-nuv[-1]
    nuv=nuv+nu_0

    return(rv,mv,nuv,pv,rhov,Gammav)

def gama_func(tov_func_array,omega):

    def functions_inside(t,y,tov_funcs,ell,omega):
        if(t==0.0):
            return([0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0])
        else:
            r=t
            H1_real,K_real,W_real,X_real,H1_imag,K_imag,W_imag,X_imag=y
            H1=H1_real+(1j*H1_imag)
            K=K_real+(1j*K_imag)
            W=W_real+(1j*W_imag)
            X=X_real+(1j*X_imag)
            m=tov_funcs[0](r)
            nu=tov_funcs[1](r)
            p=tov_funcs[2](r)
            rho=tov_funcs[3](r)
            gam=tov_funcs[4](r)
            lmb=-log(1.0-((2.0*m)/r))
            mp=4.0*pi*(r**2.0)*rho
            nup=(2.0*(m+(4.0*pi*(r**3.0)*p)))/(r*(r-(2.0*m)))
            pp=-((rho+p)*(m+(4.0*pi*(r**3.0)*p)))/(r*(r-(2.0*m)))
            lmbp=(2.0*((4.0*pi*(r**3.0)*rho)-m))/(r*(r-(2.0*m)))
            nupp=(2.0*(mp+(12.0*pi*(r**2.0)*p)+(4.0*pi*(r**3.0)*pp)-(r*(1.0-(m/r)-mp)*nup)))/(r*(r-(2.0*m)))
            H0=(((8.0*pi*(r**3.0)*exp(-nu/2.0))*X)-((((1.0/2.0)*ell*(ell+1.0)*(m+(4.0*pi*(r**3.0)*p)))-((omega**2.0)*(r**3.0)*exp(-(lmb+nu))))*H1)+((((1.0/2.0)*(ell+2.0)*(ell-1.0)*r)-((omega**2.0)*(r**3.0)*exp(-nu))-((1.0/r)*exp(lmb)*(m+(4.0*pi*(r**3.0)*p))*((3.0*m)-r+(4.0*pi*(r**3.0)*p))))*K))/((3.0*m)+((1.0/2.0)*(ell+2.0)*(ell-1.0)*r)+(4.0*pi*(r**3.0)*p))
            V=(X+(((1.0/r)*pp*exp((nu-lmb)/2.0))*W)-((1.0/2.0)*(rho+p)*exp(nu/2.0)*H0))/((omega**2.0)*(rho+p)*exp(-nu/2.0))
            dH1dr=-((1.0/r)*(ell+1.0+((2.0*m*exp(lmb))/r)+(4.0*pi*(r**2.0)*exp(lmb)*(p-rho)))*H1)+((1.0/r)*exp(lmb)*(H0+K-(16.0*pi*(rho+p)*V)))
            dKdr=((1.0/r)*H0)+((1.0/2.0)*ell*(ell+1.0)*(1.0/r)*H1)-((((ell+1.0)/r)-((1.0/2.0)*nup))*K)-(8.0*pi*(rho+p)*exp(lmb/2.0)*(1.0/r)*W)
            dWdr=-((ell+1.0)*(1.0/r)*W)+(r*exp(lmb/2.0)*(((1.0/gam)*(1.0/p)*(1.0/exp(nu/2.0))*X)-(((ell*(ell+1.0))/(r**2.0))*V)+(H0/2.0)+K))
            dXdr=-((ell*X)/r)+((rho+p)*exp(nu/2.0)*(((1.0/2.0)*((1.0/r)-((1.0/2.0)*nup))*H0)+((1.0/2.0)*((r*(omega**2.0)*exp(-nu))+((1.0/2.0)*ell*(ell+1.0)*(1.0/r)))*H1)+((1.0/2.0)*(((3.0/2.0)*nup)-(1.0/r))*K)-((1.0/2.0)*ell*(ell+1.0)*nup*(1.0/(r**2.0))*V)-((1.0/r)*((4.0*pi*(rho+p)*exp(lmb/2.0))+((omega**2.0)*exp((lmb/2.0)-nu))-((1.0/2.0)*(r**2.0)*(((1.0/(r**2.0))*nupp*exp(-lmb/2.0))-((2.0/(r**3.0))*nup*exp(-lmb/2.0))-((1.0/(2.0*(r**2.0)))*lmbp*nup*exp(-lmb/2.0)))))*W)))
            dydt=[dH1dr.real,dKdr.real,dWdr.real,dXdr.real,dH1dr.imag,dKdr.imag,dWdr.imag,dXdr.imag]
            return(dydt)

    def solver_inside(t_i,t_f,y_0,tov_funcs,ell,omega):
        t_span=(t_i,t_f)
        t_eval=linspace(t_i,t_f,5000)
        sol=solve_ivp(functions_inside,t_span,y_0,method='LSODA',t_eval=t_eval,max_step=1.0e-03,dense_output=True,args=(tov_funcs,ell,omega))
        r_vec=sol.t
        H1_vec_real=sol.y[0,:]
        K_vec_real=sol.y[1,:]
        W_vec_real=sol.y[2,:]
        X_vec_real=sol.y[3,:]
        H1_vec_imag=sol.y[4,:]
        K_vec_imag=sol.y[5,:]
        W_vec_imag=sol.y[6,:]
        X_vec_imag=sol.y[7,:]
        return(r_vec,H1_vec_real,K_vec_real,W_vec_real,X_vec_real,H1_vec_imag,K_vec_imag,W_vec_imag,X_vec_imag)

    def functions_outside(t,y,m_s,ell,omega):
        r=t
        Z_real,Zp_real,Z_imag,Zp_imag=y
        Z=Z_real+(1j*Z_imag)
        Zp=Zp_real+(1j*Zp_imag)
        nn=(1.0/2.0)*(ell-1.0)*(ell+2.0)
        pot=((1.0-((2.0*m_s)/r))*((2.0*(nn**2.0)*(nn+1.0)*(r**3.0))+(6.0*(nn**2.0)*m_s*(r**2.0))+(18.0*nn*(m_s**2.0)*r)+(18.0*(m_s**3.0))))/((r**3.0)*(((nn*r)+(3.0*m_s))**2.0))
        dZdr=Zp
        ddZdrr=-((1.0/(1.0-((2.0*m_s)/r)))*((2.0*m_s)/(r**2.0))*Zp)-(((1.0/(1.0-((2.0*m_s)/r)))**2.0)*((omega**2.0)-pot)*Z)
        dydt=[dZdr.real,ddZdrr.real,dZdr.imag,ddZdrr.imag]
        return(dydt)

    def solver_outside(t_i,t_f,y_0,m_s,ell,omega):
        t_span=(t_i,t_f)
        t_eval=linspace(t_i,t_f,10000)
        sol=solve_ivp(functions_outside,t_span,y_0,method='LSODA',t_eval=t_eval,max_step=1.0e-02,dense_output=True,args=(m_s,ell,omega))
        r_vec=sol.t
        Z_vec_real=sol.y[0,:]
        Zp_vec_real=sol.y[1,:]
        Z_vec_imag=sol.y[2,:]
        Zp_vec_imag=sol.y[3,:]
        return(r_vec,Z_vec_real,Zp_vec_real,Z_vec_imag,Zp_vec_imag)

    rv,mv,nuv,pv,rhov,Gammav=tov_func_array

    ######################
    # initial conditions # 
    ######################

    r_0=rv[0]
    r_s=rv[-1]
    nu_0=nuv[0]
    p_0=pv[0]
    rho_0=rhov[0]

    #################################################
    # creates interpolation function for tov arrays #
    #################################################

    m_tov=CubicSpline(rv,mv)
    nu_tov=CubicSpline(rv,nuv)
    p_tov=CubicSpline(rv,pv)
    rho_tov=CubicSpline(rv,rhov)
    Gamma_tov=CubicSpline(rv,Gammav)
    tov_funcs=[m_tov,nu_tov,p_tov,rho_tov,Gamma_tov]

    #################
    # choice of ell #
    #################

    ell=2.0

    ######################
    # forward solution 1 #
    ######################

    K_0=+(rho_0+p_0)
    W_0=1.0
    X_0=(rho_0+p_0)*exp(nu_0/2.0)*((((((4.0*pi)/3.0)*(rho_0+(3.0*p_0)))-((omega**2.0)/(exp(nu_0)*ell)))*W_0)+((1.0/2.0)*K_0))
    H1_0=((2.0*ell*K_0)+((16.0*pi*(rho_0+p_0))*W_0))/(ell*(ell+1.0))

    y_0=[H1_0.real,K_0.real,W_0.real,X_0.real,H1_0.imag,K_0.imag,W_0.imag,X_0.imag]

    all_vars=solver_inside(r_0,0.5*r_s,y_0,tov_funcs,ell,omega)

    H14_vec=all_vars[1]+(1j*all_vars[5])
    K4_vec=all_vars[2]+(1j*all_vars[6])
    W4_vec=all_vars[3]+(1j*all_vars[7])
    X4_vec=all_vars[4]+(1j*all_vars[8])

    r4_vec=all_vars[0]
    H14=H14_vec[-1]
    K4=K4_vec[-1]
    W4=W4_vec[-1]
    X4=X4_vec[-1]

    ######################
    # forward solution 2 #
    ######################

    K_0=-(rho_0+p_0)
    W_0=1.0
    X_0=(rho_0+p_0)*exp(nu_0/2.0)*((((((4.0*pi)/3.0)*(rho_0+(3.0*p_0)))-((omega**2.0)/(exp(nu_0)*ell)))*W_0)+((1.0/2.0)*K_0))
    H1_0=((2.0*ell*K_0)+((16.0*pi*(rho_0+p_0))*W_0))/(ell*(ell+1.0))

    y_0=[H1_0.real,K_0.real,W_0.real,X_0.real,H1_0.imag,K_0.imag,W_0.imag,X_0.imag]

    all_vars=solver_inside(r_0,0.5*r_s,y_0,tov_funcs,ell,omega)

    H15_vec=all_vars[1]+(1j*all_vars[5])
    K5_vec=all_vars[2]+(1j*all_vars[6])
    W5_vec=all_vars[3]+(1j*all_vars[7])
    X5_vec=all_vars[4]+(1j*all_vars[8])

    H15=H15_vec[-1]
    K5=K5_vec[-1]
    W5=W5_vec[-1]
    X5=X5_vec[-1]

    #######################
    # backward solution 1 #
    #######################

    H1_s=1.0
    K_s=0.0
    W_s=0.0
    X_s=0.0

    y_s=[H1_s.real,K_s.real,W_s.real,X_s.real,H1_s.imag,K_s.imag,W_s.imag,X_s.imag]

    all_vars=solver_inside(r_s,0.5*r_s,y_s,tov_funcs,ell,omega)

    r1_vec=all_vars[0]
    H11_vec=all_vars[1]+(1j*all_vars[5])
    K1_vec=all_vars[2]+(1j*all_vars[6])
    W1_vec=all_vars[3]+(1j*all_vars[7])
    X1_vec=all_vars[4]+(1j*all_vars[8])

    H11=H11_vec[-1]
    K1=K1_vec[-1]
    W1=W1_vec[-1]
    X1=X1_vec[-1]

    #######################
    # backward solution 2 #
    #######################

    H1_s=0.0
    K_s=1.0
    W_s=0.0
    X_s=0.0

    y_s=[H1_s.real,K_s.real,W_s.real,X_s.real,H1_s.imag,K_s.imag,W_s.imag,X_s.imag]

    all_vars=solver_inside(r_s,0.5*r_s,y_s,tov_funcs,ell,omega)

    H12_vec=all_vars[1]+(1j*all_vars[5])
    K2_vec=all_vars[2]+(1j*all_vars[6])
    W2_vec=all_vars[3]+(1j*all_vars[7])
    X2_vec=all_vars[4]+(1j*all_vars[8])

    H12=H12_vec[-1]
    K2=K2_vec[-1]
    W2=W2_vec[-1]
    X2=X2_vec[-1]

    #######################
    # backward solution 3 #
    #######################

    H1_s=0.0
    K_s=0.0
    W_s=1.0
    X_s=0.0

    y_s=[H1_s.real,K_s.real,W_s.real,X_s.real,H1_s.imag,K_s.imag,W_s.imag,X_s.imag]

    all_vars=solver_inside(r_s,0.5*r_s,y_s,tov_funcs,ell,omega)

    H13_vec=all_vars[1]+(1j*all_vars[5])
    K3_vec=all_vars[2]+(1j*all_vars[6])
    W3_vec=all_vars[3]+(1j*all_vars[7])
    X3_vec=all_vars[4]+(1j*all_vars[8])

    H13=H13_vec[-1]
    K3=K3_vec[-1]
    W3=W3_vec[-1]
    X3=X3_vec[-1]

    ################################################
    # creates arrays for solutions inside the star #
    ################################################

    arr_A=array([[H11,H12,H13,H14],[K1,K2,K3,K4],[W1,W2,W3,W4],[X1,X2,X3,X4]])
    arr_B=array([H15,K5,W5,X5])
    arr_inv_A=linalg.inv(arr_A)
    arr_a=arr_inv_A.dot(arr_B)

    a_1=arr_a[0]
    a_2=arr_a[1]
    a_3=arr_a[2]
    a_4=arr_a[3]

    r_vec_2=r1_vec
    H1_vec_2=(a_1*H11_vec)+(a_2*H12_vec)+(a_3*H13_vec)
    K_vec_2=(a_1*K1_vec)+(a_2*K2_vec)+(a_3*K3_vec)
    W_vec_2=(a_1*W1_vec)+(a_2*W2_vec)+(a_3*W3_vec)
    X_vec_2=(a_1*X1_vec)+(a_2*X2_vec)+(a_3*X3_vec)

    r_vec_1=r4_vec
    H1_vec_1=-(a_4*H14_vec)+H15_vec
    K_vec_1=-(a_4*K4_vec)+K5_vec
    W_vec_1=-(a_4*W4_vec)+W5_vec
    X_vec_1=-(a_4*X4_vec)+X5_vec

    r_vec=concatenate((r_vec_1,r_vec_2[::-1]))
    H1_vec=concatenate((H1_vec_1,H1_vec_2[::-1]))
    K_vec=concatenate((K_vec_1,K_vec_2[::-1])) 
    W_vec=concatenate((W_vec_1,W_vec_2[::-1]))
    X_vec=concatenate((X_vec_1,X_vec_2[::-1]))

    ######################
    # updates tov arrays #
    ######################

    m_vec=[]
    nu_vec=[]
    p_vec=[]
    rho_vec=[]
    Gamma_vec=[]
    for i in range(len(r_vec)):
        m_vec.append(m_tov(r_vec[i]))
        nu_vec.append(nu_tov(r_vec[i]))
        p_vec.append(p_tov(r_vec[i]))
        rho_vec.append(rho_tov(r_vec[i]))
        Gamma_vec.append(Gamma_tov(r_vec[i]))
    m_vec=array(m_vec)
    nu_vec=array(nu_vec)
    p_vec=array(p_vec)
    rho_vec=array(rho_vec)
    Gamma_vec=array(Gamma_vec)

    ##########
    # arrays #
    ##########

    rv=r_vec
    mv=m_vec
    nuv=nu_vec
    pv=p_vec
    rhov=rho_vec
    Gammav=Gamma_vec
    H1_real=H1_vec.real
    K_real=K_vec.real
    W_real=W_vec.real
    X_real=X_vec.real
    H1_imag=H1_vec.imag
    K_imag=K_vec.imag
    W_imag=W_vec.imag
    X_imag=X_vec.imag

    ######################
    # initial conditions # 
    ######################

    r_s=rv[-1]
    m_s=mv[-1]
    nu_s=nuv[-1]
    p_s=pv[-1]
    H1_out_s=H1_real[-1]+(1j*H1_imag[-1])
    K_out_s=K_real[-1]+(1j*K_imag[-1])
    X_out_s=X_real[-1]+(1j*X_imag[-1])
    lmb_s=-log(1.0-((2.0*m_s)/r_s))

    Ks=K_out_s
    H0s=(((8.0*pi*(r_s**3.0)*exp(-nu_s/2.0))*X_out_s)-((((1.0/2.0)*ell*(ell+1.0)*(m_s+(4.0*pi*(r_s**3.0)*p_s)))-((omega**2.0)*(r_s**3.0)*exp(-(lmb_s+nu_s))))*H1_out_s)+((((1.0/2.0)*(ell+2.0)*(ell-1.0)*r_s)-((omega**2.0)*(r_s**3.0)*exp(-nu_s))-((1.0/r_s)*exp(lmb_s)*(m_s+(4.0*pi*(r_s**3.0)*p_s))*((3.0*m_s)-r_s+(4.0*pi*(r_s**3.0)*p_s))))*K_out_s))/((3.0*m_s)+((1.0/2.0)*(ell+2.0)*(ell-1.0)*r_s)+(4.0*pi*(r_s**3.0)*p_s))

    ##########################
    # solves the zerilli eqs #
    ##########################

    nn=(1.0/2.0)*(ell-1.0)*(ell+2.0)
    aas=-((nn*r_s)+(3.0*m_s))/(((omega**2.0)*(r_s**2.0))-(((nn+1.0)*m_s)/r_s))
    bbs=((nn*r_s*(r_s-(2.0*m_s)))-((omega**2.0)*(r_s**4.0))+(m_s*(r_s-(3.0*m_s))))/((r_s-(2.0*m_s))*(((omega**2.0)*(r_s**2.0))-(((nn+1.0)*m_s)/r_s)))
    ggs=((nn*(nn+1.0)*(r_s**2.0))+(3.0*nn*m_s*r_s)+(6.0*(m_s**2.0)))/((r_s**2.0)*((nn*r_s)+(3.0*m_s)))
    hhs=((-nn*(r_s**2.0))+(3.0*nn*m_s*r_s)+(3.0*(m_s**2.0)))/((r_s-(2.0*m_s))*((nn*r_s)+(3.0*m_s)))
    kks=-(r_s**2.0)/(r_s-(2.0*m_s))

    Zs=((kks*Ks)-(aas*H0s)-(bbs*Ks))/((kks*ggs)-hhs)
    Zps=(1.0/(1.0-((2.0*m_s)/r_s)))*(((hhs*Ks)-(aas*ggs*H0s)-(bbs*ggs*Ks))/(hhs-(kks*ggs)))

    a=25.0/omega.real

    y_0=[Zs.real,Zps.real,Zs.imag,Zps.imag]
    all_vars=solver_outside(r_s,a,y_0,m_s,ell,omega)

    r_out_vec=all_vars[0]
    Z_vec_real=all_vars[1]
    Zp_vec_real=all_vars[2]
    Z_vec_imag=all_vars[3]
    Zp_vec_imag=all_vars[4]

    Z_vec=Z_vec_real+(1j*Z_vec_imag)
    Zp_vec=Zp_vec_real+(1j*Zp_vec_imag)

    Zf=Z_vec[-1]
    Zpf=Zp_vec[-1]

    beta0=1.0
    beta1=-(1j*(nn+1.0)*beta0)/omega
    beta2=-(((nn*(nn+1.0))-(3.0*1j*m_s*omega*(1.0+(2.0/nn))))*beta0)/(2.0*(omega**2.0))

    r=a
    rs=r+(2.0*m_s*log((r/(2.0*m_s))-1.0))
    Zminus=exp(-1j*omega*rs)*(beta0+(beta1/r)+(beta2/(r**2.0)))
    Zplus=conj(Zminus)
    Zpminus=-((1j*omega*Zminus)/(1.0-((2.0*m_s)/r)))-(exp(-1j*omega*rs)*((beta1/(r**2.0))+((2.0*beta2)/(r**3.0))))
    Zpplus=conj(Zpminus)

    Z_11=Zminus
    Z_12=Zplus
    Z_21=Zpminus
    Z_22=Zpplus
    detZ=(Z_11*Z_22)-(Z_12*Z_21)

    Zinv_11=Z_22/detZ
    Zinv_12=-Z_12/detZ
    Zinv_21=-Z_21/detZ
    Zinv_22=Z_11/detZ

    beta=(Zinv_11*Zf)+(Zinv_12*Zpf)
    gama=(Zinv_21*Zf)+(Zinv_22*Zpf)

    return(gama)

def muller_func(omega_1,omega_2,omega_3,gama_1,gama_2,gama_3):

    if((abs(gama_1)>abs(gama_2))and(abs(gama_1)>abs(gama_3))):
        if(abs(gama_2)>abs(gama_3)):
            z_1=omega_1
            z_2=omega_2
            z_3=omega_3
            f_1=gama_1
            f_2=gama_2
            f_3=gama_3
        elif(abs(gama_3)>abs(gama_2)):
            z_1=omega_1
            z_2=omega_3
            z_3=omega_2
            f_1=gama_1
            f_2=gama_3
            f_3=gama_2
    elif((abs(gama_2)>abs(gama_3))and(abs(gama_2)>abs(gama_1))):
        if(abs(gama_1)>abs(gama_3)):
            z_1=omega_2
            z_2=omega_1
            z_3=omega_3
            f_1=gama_2
            f_2=gama_1
            f_3=gama_3
        elif(abs(gama_3)>abs(gama_1)):
            z_1=omega_2
            z_2=omega_3
            z_3=omega_1
            f_1=gama_2
            f_2=gama_3
            f_3=gama_1
    elif((abs(gama_3)>abs(gama_1))and(abs(gama_3)>abs(gama_2))):
        if(abs(gama_1)>abs(gama_2)):
            z_1=omega_3
            z_2=omega_1
            z_3=omega_2
            f_1=gama_3
            f_2=gama_1
            f_3=gama_2
        elif(abs(gama_2)>abs(gama_1)):
            z_1=omega_3
            z_2=omega_2
            z_3=omega_1
            f_1=gama_3
            f_2=gama_2
            f_3=gama_1

    q=(z_3-z_2)/(z_2-z_1)
    aa=(q*f_3)-(q*(1.0+q)*f_2)+((q**2.0)*f_1)
    bb=(((2.0*q)+1.0)*f_3)-(((1.0+q)**2.0)*f_2)+((q**2.0)*f_1)
    cc=(1.0+q)*f_3
    dd=(bb**2.0)-(4.0*aa*cc)

    if(abs(bb+sqrt(dd))>abs(bb-sqrt(dd))):
        z_4=z_3-((z_3-z_2)*((2.0*cc)/(bb+sqrt(dd))))
    else:
        z_4=z_3-((z_3-z_2)*((2.0*cc)/(bb-sqrt(dd))))

    f_abs=abs(f_3)

    omega_1=z_2
    omega_2=z_3
    omega_3=z_4

    return(omega_1,omega_2,omega_3,f_abs)

def shooting_func(omega_guess):

    omega_1=omega_guess*0.9
    omega_2=omega_guess*1.0
    omega_3=omega_guess*1.1

    gama_1=gama_func(tov_func_array,omega_1)
    gama_2=gama_func(tov_func_array,omega_2)
    gama_3=gama_func(tov_func_array,omega_3)

    tol=1.0e-08

    omega_r_bf=omega_guess.real
    delta_abs=1.0

    while(delta_abs>tol):

        omega_1,omega_2,omega_3,gama_abs=muller_func(omega_1,omega_2,omega_3,gama_1,gama_2,gama_3)

        gama_1=gama_func(tov_func_array,omega_1)
        gama_2=gama_func(tov_func_array,omega_2)
        gama_3=gama_func(tov_func_array,omega_3)

        omega_r_af=omega_3
        delta_abs=abs(1.0-(omega_r_af/omega_r_bf))
        omega_r_bf=omega_r_af

        print(omega_3,delta_abs)

    omega_final=omega_3
    
    return(omega_final)

##############################################
# initial condition: eos and central density #
##############################################

eos_str='SLy4'
rho_0_cgs=9.87e+14

#################################
# mass, radius, and compactness #
#################################

eos_func_array=eos_func(eos_str)
tov_func_array=tov_func(eos_func_array,rho_0_cgs)
R=tov_func_array[0][-1]
M=tov_func_array[1][-1]
C=M/R

############################################
# guess for the QNM frequency using TL fit #
############################################

# f-mode fit params
a_TL=0.15-(1j*5.8e-04)
b_TL=0.56+(1j*6.7e-04)
c_TL=-0.020-(1j*6.2e-05)

omega_guess=(1.0/M)*((a_TL*(C**2.0))+(b_TL*C)+c_TL)
print('QNM guess:',omega_guess)
print('---')

#################
# QNM frequency #
#################

omega=shooting_func(omega_guess)
print('---')
print('QNM final:',omega)


QNM guess: (0.04038565846380339+1.8466313982815966e-05j)
---
(0.0405105119257618+1.7088534236089936e-05j) 0.0031203522141849144
(0.040515956434424974+1.733665625455638e-05j) 0.0001345369092100301
(0.040515946967213216+1.7336921087132908e-05j) 2.3375763944369643e-07
(0.04051596803650501+1.7335587536099636e-05j) 5.210652070889499e-07
(0.04051595657161698+1.7346600857250453e-05j) 3.923810886303313e-07
(0.04051595414909284+1.7342659059897593e-05j) 1.141946021251173e-07
(0.04051595585443913+1.7338192809365454e-05j) 1.1799679208010258e-07
(0.04051595652477804+1.7336158410946342e-05j) 5.286786646784174e-08
(0.04051595646414122+1.733541816930184e-05j) 1.83315665703961e-08
(0.04051595565793149+1.733593850664013e-05j) 2.3683116281800988e-08
(0.04051595655344457+1.7336236289146894e-05j) 2.3292689199097118e-08
(0.04051595646146607+1.7336068871786612e-05j) 4.7146836682814646e-09
---
QNM final: (0.04051595646146607+1.7336068871786612e-05j)
